**EEG–Eye-Tracking Multimodal Research | Presented by: Reza Saadatyar (2026)**

**03 | EEG and Gaze Preprocessing, Features and Neurophysiology**

![Feature extraction from recorded paired ZuCo windows](../../images/03_preprocessing_features_and_neurophysiology.svg)

**Research question:** Can comparable EEG-only and gaze-only feature representations be extracted from the same eligible BIDS-relative windows without losing participant/task alignment?

**Objective:** Apply separate modality preprocessing, extract spectral EEG and gaze/event features, save matched six-step temporal representations, and produce a descriptive Train-only fixation-locked EEG example.

**Recorded upstream inputs:** Notebook 02 produced **206 eligible 12-second windows** from **12 NR/TSR runs** and **six participants**. These comprise **95 Train, 31 Validation and 80 Test** windows. The saved 12-run quality report does not verify independent EEG/eye clock drift.

**Feature design:** A fixed set of up to **16 actual EEG channels**, 1–40 Hz bandpass, six 2-second bins per window, four EEG descriptors and four gaze descriptors per bin. Later models perform any learned imputation or scaling using Train only.

**Scientific boundary:** This notebook extracts **descriptive paired signals**, not cognitive or clinical predictions. Stage-02 x/y coverage is an eligibility check, not proof of pupil quality, absence of ocular artefacts or physiologically exact EEG–eye synchronisation.


**00. Colab Repository Setup**

In [ ]:
# ======================== 00. Colab Repository Setup =========================

# -------------------------- 1. Repository Checkout ---------------------------
from pathlib import Path
import subprocess

repo_url = "https://github.com/RezaSaadatyar/Eye-Tracking-Neurocognitive-Intelligence.git"
repo_path = Path("/content") / repo_url.rsplit("/", 1)[-1].removesuffix(".git")
if not repo_path.exists():
    subprocess.run(
        ["git", "clone", "-q", repo_url, str(repo_path)], check=True
    )

# ---------------------------- 2. Open Repository -----------------------------
%cd {repo_path}


**01. Imports and Research Environment**

In [ ]:
# =================== 01. Imports and Research Environment ====================

# ------------------------ 1. Study Environment Setup -------------------------
from python_modules.core.colab_setup import setup_colab
root = setup_colab("eeg_eye_tracking_fusion")

# ---------------------------- 2. Standard Library ----------------------------
import importlib.util
import json
import random
import shutil
import subprocess
import sys
from pathlib import Path

# ---------------------- 3. Colab Dependency Preparation ----------------------
if importlib.util.find_spec("mne") is None:
    subprocess.run(
        [sys.executable, "-m", "pip", "-q", "install",
         "mne>=1.6,<2"],
        check=True,
    )

# ----------------- 4. Scientific Computing and Data Analysis -----------------
import numpy as np
import pandas as pd
from scipy.signal import butter, sosfiltfilt

# ------------------- 5. Visualisation and Notebook Display -------------------
import matplotlib.pyplot as plt
from IPython.display import display
from core.plot_style import plot_style

# ----------------- 6. Shared and Project Research Utilities ------------------
from core.output_download import download_outputs_from_colab
from zuco_io import open_run, session
from alignment import ensure_disjoint_registry
from features import (
    EEG_FIELDS, GAZE_FIELDS, eeg_channels, extract_window, clean_eeg,
)

# ---------------------------- 7. Reproducibility -----------------------------
SEED = 17
random.seed(SEED)
np.random.seed(SEED)

# ------------------- 8. Output Directory and Setup Summary -------------------
work = root / "eeg_eye_tracking_fusion"
output_dir = work / "output"
output_dir.mkdir(parents=True, exist_ok=True)
raw_dir = Path("/content/zuco2_source_cache")
raw_dir.mkdir(parents=True, exist_ok=True)
plot_style(font_family="DejaVu Sans", figure_dpi=100)
print("Project outputs:", output_dir)
print("Temporary source cache:", raw_dir)
print("Python:", sys.version.split()[0])
print("Free disk (GiB):", round(
    shutil.disk_usage("/content").free / 2**30, 2,
))


**02. Load and Verify Paired-Window Inputs**

Read the current `01_source_manifest.csv`, `01_protocol.json`, `02_window_registry.csv` and `02_alignment_quality.csv`. Use only the windows eligible under the fixed 12-second and finite-gaze-coordinate protocol.

**Upstream evidence:** **206 eligible windows**, 101 NR and 105 TSR, from six independent people. Keep the original participant, task, Run, split and `window_id` fields unchanged; report the recorded support for each group.

The documented alignment uses BIDS `StartTime` and sample frequencies, and `drift_verified` remains `False`. No independently validated clock correction is applied.


In [ ]:
# ============= 02. Verify Paired Windows and Modality Contracts ==============

required = [
    "01_source_manifest.csv", "01_protocol.json",
    "02_window_registry.csv", "02_alignment_quality.csv",
]
missing = [p for p in required if not (output_dir / p).is_file()]
if missing:
    raise FileNotFoundError(
        f"Missing upstream source or alignment outputs: {missing}"
    )

source = pd.read_csv(
    output_dir / "01_source_manifest.csv",
    dtype={"participant_id": str},
)
registry = pd.read_csv(
    output_dir / "02_window_registry.csv",
    dtype={"participant_id": str},
)
alignment_quality = pd.read_csv(
    output_dir / "02_alignment_quality.csv",
    dtype={"participant_id": str},
)
protocol = json.loads((output_dir / "01_protocol.json").read_text())

ensure_disjoint_registry(registry)
selected = registry.loc[registry.eligible].copy()
if selected.empty:
    raise ValueError("No eligible paired windows from Notebook 02")

# A selected window must belong to its original source run and split.
join_keys = ["participant_id", "task", "run"]
source_roles = source[join_keys + ["split"]].drop_duplicates()
linked = selected.merge(
    source_roles, on=join_keys, how="left",
    suffixes=("_window", "_source"), validate="many_to_one",
)
if (linked.split_source.isna().any() or not
        linked.split_window.eq(linked.split_source).all()):
    raise ValueError("Window source identities or participant roles differ")

window_length = float(protocol["window_seconds"])
bin_seconds = float(protocol["bin_seconds"])
bins_ratio = window_length / bin_seconds
if not np.isclose(bins_ratio, round(bins_ratio)):
    raise ValueError("Window length must divide into equal feature bins")
BINS = int(round(bins_ratio))
REQUEST = session()

print("Recorded source participants:", source.participant_id.nunique())
print("Recorded original runs:", len(source))
print("Eligible paired windows:", len(selected))
print("Feature bins per window:", BINS)
print("Independent clock drift verified:",
      alignment_quality.drift_verified.astype(str).str.lower().eq(
          "true").all())
display(
    selected.groupby(["split", "task"])
    .agg(windows=("window_id", "size"),
         participants=("participant_id", "nunique"))
    .reset_index()
)


**03. Extract Modality-Separated Features on Identical Windows**

For each eligible 12-second window, read the selected EEG channels and matching EyeLink samples. EEG is average-referenced within the selected channels and bandpass filtered at **1–40 Hz**. Six two-second bins yield theta/alpha/beta log bandpower and EEG RMS. Gaze bins yield finite-coordinate coverage, median pixel speed, pupil median and fixation-event occupancy.

**Data governance:** Features remain linked to the same `window_id`; sample-level missingness is retained rather than concealed by globally fitted imputation. Later modelling fits any learned transformations on **Train participants only**. Report per-Run feature successes and failures, particularly the single-window `sub-YAK / NR` source.


In [ ]:
# ============ 03. Extract Matched Modality-Separated Features ================

feature_rows, signal_windows, gaze_windows, accepted, issues = (
    [], [], [], [], [])
for _, run in source.iterrows():
    subset = selected.loc[
        selected.participant_id.eq(run.participant_id) &
        selected.task.eq(run.task) &
        selected.run.eq(run.run)]
    if subset.empty:
        continue
    preceding_ok = len(accepted)
    preceding_issues = len(issues)
    raw, eye, events, eye_meta, _ = open_run(run, raw_dir, REQUEST)
    channels = eeg_channels(raw, count=16)
    try:
        for _, row in subset.iterrows():
            try:
                eeg_seq, gaze_seq = extract_window(
                    raw, eye, events, eye_meta, row, channels,
                    bin_seconds=float(protocol['bin_seconds']))
                if eeg_seq.shape != (BINS, len(EEG_FIELDS)):
                    raise ValueError('EEG feature shape mismatch')
                if gaze_seq.shape != (BINS, len(GAZE_FIELDS)):
                    raise ValueError('Gaze feature shape mismatch')
                fields = {name: float(np.nanmean(eeg_seq[:, j]))
                          for j, name in enumerate(EEG_FIELDS)}
                fields.update({
                    name: float(np.nanmean(gaze_seq[:, j]))
                    for j, name in enumerate(GAZE_FIELDS)})
                feature_rows.append({**row.to_dict(), **{
                    'eeg_' + name: value
                    for name, value in fields.items()
                    if name in EEG_FIELDS}, **{
                    'gaze_' + name: value
                    for name, value in fields.items()
                    if name in GAZE_FIELDS}})
                signal_windows.append(eeg_seq)
                gaze_windows.append(gaze_seq)
                accepted.append(str(row.window_id))
            except (ValueError, FloatingPointError) as exc:
                issues.append({'window_id': row.window_id,
                               'reason': str(exc)})
    finally:
        raw.close()
    print(
        "Processed:", run.participant_id, run.task,
        "| candidate windows:", len(subset),
        "| retained:", len(accepted) - preceding_ok,
        "| excluded:", len(issues) - preceding_issues,
    )
features = pd.DataFrame(feature_rows)
if features.empty:
    raise RuntimeError('No paired temporal features were produced')
if features.window_id.duplicated().any():
    raise ValueError("Feature extraction duplicated original window IDs")
print("Extracted paired feature windows:", len(features))
print("Feature extraction issues:", len(issues))
display(features.head())


**04. Inspect Feature Coverage and Participant Support**

Summarise successful feature windows by participant, split and source task. Show the number and reasons for any feature-extraction exclusions.

An original participant must retain both NR and TSR support after feature extraction; a missing task or split indicates that the intended participant-disjoint comparison is no longer valid. These scientific requirements produce explicit error messages rather than fixed-count assertions.

**Interpretation:** Multiple windows from one person are correlated. Class counts and results must not be interpreted as independent-person sample sizes.


In [ ]:
# ============ 04. Check EEG/Gaze Completeness and Split Coverage =============

quality = pd.DataFrame(issues, columns=['window_id', 'reason'])
group_support = (features.groupby(['participant_id', 'split', 'task'])
                 .size().rename('n_windows').reset_index())
eligible_people = features.groupby('participant_id').task.nunique()
if not eligible_people.eq(2).all():
    raise ValueError('A participant lost one task after processing')
if set(features.split) != {'train', 'validation', 'test'}:
    raise ValueError('A split role is absent from processed windows')
display(group_support)
print("Excluded after feature extraction:", len(quality))
if not quality.empty:
    display(quality.groupby("reason").size().rename("n_windows"))
eeg_fields = ["eeg_" + name for name in EEG_FIELDS]
gaze_fields = ["gaze_" + name for name in GAZE_FIELDS]
print("Missing/non-finite EEG summary values:",
      int((~np.isfinite(features[eeg_fields])).sum().sum()))
print("Missing/non-finite gaze summary values:",
      int((~np.isfinite(features[gaze_fields])).sum().sum()))


**05. Descriptive Fixation-Locked EEG Diagnostic (Train Only)**

Compute a small illustrative fixation-onset average from a single Train recording, using the original EEG signal and recorded fixation events. Display the selected channel signal in microvolts around event onset.

The example is **exploratory and descriptive**: it does not apply a formal artefact-corrected or deconvolved FRP model, validate synchrony, localise a cortical source or explain a cognitive mechanism. It is not included as a predictive model feature.


In [ ]:
# ======== 05. Fixation-Locked EEG: Descriptive Train-Only Diagnostic =========


training_run = source.loc[source.split.eq('train')].iloc[0]
raw, eye, events, eye_meta, _ = open_run(
    training_run, raw_dir, REQUEST)
try:
    chans = eeg_channels(raw, count=16)
    starts = pd.to_numeric(
        events.loc[events.trial_type.eq('fixation'), 'onset'],
        errors='coerce').dropna().to_numpy()
    starts = starts[(starts > 0.2) &
                    (starts < raw.times[-1] - 0.6)][:80]
    sf = float(raw.info['sfreq'])
    samples = []
    for onset in starts:
        seg = raw.get_data(picks=chans[:1],
                           start=int((onset-0.2)*sf),
                           stop=int((onset+0.6)*sf))[0]
        if len(seg) == int(0.8 * sf) and np.isfinite(seg).all():
            samples.append(seg)
    fig, ax = plt.subplots(figsize=(7, 3.5))
    if samples:
        epochs = np.stack(samples)
        avg = np.mean(epochs, axis=0)
        t = np.arange(len(avg))/sf - 0.2
        ax.plot(t, avg*1e6)
        ax.axvline(0, color='black', ls='--', lw=1)
    ax.set(title='Exploratory fixation-locked EEG (Train only)',
           xlabel='Seconds from fixation onset', ylabel='EEG (µV)')
    fig.tight_layout()
    fig.savefig(output_dir / '03_fixation_locked_diagnostic.png',
                dpi=170)
    plt.show()
finally:
    raw.close()


**06. Save Matched Features and Compact Temporal Arrays**

Save the accepted original windows in the same ordered `window_id` sequence for both modalities:

- `03_features.csv`: paired window-level EEG/gaze feature summaries and source identities.
- `03_temporal_sequences.npz`: **six time bins × four features** for each modality, with identical window order.
- `03_exclusion_reasons.csv` and `03_modality_quality.csv`: per-window failure reasons and retained participant/task counts.
- `03_fixation_locked_diagnostic.png` and `03_temporal_example.png`: descriptive plots.

The temporal example normalises **each feature within the displayed window only**, so EEG log power/RMS and gaze speed/pupil units are not misleadingly compared on one axis. The saved NPZ and CSV retain original feature values for later Train-only preprocessing. Original EEG and eye recordings remain outside the Git repository.


In [ ]:
# ====== 06. Save Matched Features and Compact Temporal Representations =======

eeg_seq = np.stack(signal_windows).astype(np.float32)
gaze_seq = np.stack(gaze_windows).astype(np.float32)
if (len(features) != len(accepted) or
        not np.array_equal(features.window_id.astype(str), accepted)):
    raise ValueError('Feature/sequence identity mismatch')
features.to_csv(output_dir / '03_features.csv', index=False)
quality.to_csv(output_dir / '03_exclusion_reasons.csv', index=False)
group_support.to_csv(output_dir / '03_modality_quality.csv',
                     index=False)
np.savez_compressed(
    output_dir / '03_temporal_sequences.npz', eeg=eeg_seq,
    gaze=gaze_seq,
    window_id=features.window_id.astype(str).to_numpy(),
    participant_id=features.participant_id.astype(str).to_numpy(),
    task=features.task.astype(str).to_numpy(),
    split=features.split.astype(str).to_numpy())
# This display standardises each feature within one sample window.
# Saved arrays retain their original, unscaled units for later modelling.
def _display_relative_feature_trends(ax, seq, names, title):
    steps = np.arange(1, seq.shape[0] + 1)
    for col, name in enumerate(names):
        values = np.asarray(seq[:, col], dtype=float)
        available = np.isfinite(values)
        if not available.any():
            continue
        centre = np.mean(values[available])
        spread = np.std(values[available])
        relative = np.full(len(values), np.nan, dtype=float)
        relative[available] = (
            (values[available] - centre) / spread
            if spread > 0 else 0.
        )
        ax.plot(steps, relative, marker="o", label=name)
    ax.set(
        title=title,
        xlabel="2-second feature bin",
        ylabel="Within-window z-score (display only)",
        xticks=steps,
    )
    if ax.lines:
        ax.legend(fontsize=7, loc="best")


fig, axes = plt.subplots(1, 2, figsize=(12, 4.1))
_display_relative_feature_trends(
    axes[0], eeg_seq[0], EEG_FIELDS, "EEG features: one paired window"
)
_display_relative_feature_trends(
    axes[1], gaze_seq[0], GAZE_FIELDS, "Gaze features: same window"
)
fig.tight_layout()
fig.savefig(output_dir / "03_temporal_example.png", dpi=170)
plt.show()
print("Saved original-unit arrays:", eeg_seq.shape, gaze_seq.shape)


**07. Interpretation and Next Dependency**

Report the number of matched feature windows, independent Train/Validation/Test people, NR/TSR support and any incomplete or non-finite feature values. The compact EEG/eye arrays enable the same source windows to be used by classical and temporal candidates.

**Next:** Notebook 04 fits the classical unimodal and simple-fusion models on Train only and compares them on Validation. Notebook 05 examines temporal models under the same grouping; the reserved Test people must not guide parameter selection or checkpoint choice.


In [ ]:
# ================== 07. Interpretation and Next Dependency ===================

print("Stage-03 saved files:")
for path in sorted(output_dir.glob("03_*")):
    print(" ", path.name)
print("Next: 04_unimodal_baselines_and_classical_fusion.ipynb")

# ---------------------- 2. Package Stage-Owned Outputs -----------------------
download_outputs_from_colab(
    "eeg_eye_tracking_fusion/output", "03"
)
